# Comment Perception Monitor: Colab demo

Paste a YouTube link and see the sentiment, toxicity, questions and word-level explanations of its comments.

**Before running:** Runtime → Change runtime type → **T4 GPU**, and add your YouTube API key in the 🔑 Secrets panel as `YOUTUBE_API_KEY` (with notebook access on).

In [ ]:
!git clone -q https://github.com/abdulloh006/comment-perception-monitor.git
%cd comment-perception-monitor
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ["YOUTUBE_API_KEY"] = userdata.get("YOUTUBE_API_KEY")

from src.fetch.youtube import get_client, get_comments_from_url
from src.predict import analyze, get_classifier
from src.explain import explain, render_html
from IPython.display import HTML, display

youtube = get_client()

## 1. Fetch and analyze

In [ ]:
URL = "https://www.youtube.com/watch?v=2pFU6YWSggU"  # change me

df, error = get_comments_from_url(youtube, URL, max_comments=300)
if error:
    print(error)
else:
    result, info = analyze(df)  # first run downloads the models
    print(info)
    scored = result["toxicity"].notna()
    print(f"{len(result)} comments | toxic: {result.loc[scored, 'is_toxic'].mean():.0%} | questions: {result['is_question'].sum()}")
    display(result["sentiment"].value_counts())

## 2. Most toxic comments

In [ ]:
result[scored].sort_values("toxicity", ascending=False)[["text", "toxicity", "sentiment", "likes"]].head(10)

## 3. Questions viewers are asking

In [ ]:
result[result["is_question"]].sort_values("likes", ascending=False)[["text", "likes", "replies"]].head(10)

## 4. Why did the model decide that?

Red words pushed toward toxic, blue words pushed away. Try the bias pairs.

In [ ]:
clf = get_classifier("toxicity", "en")
for s in ["I am a gay man.", "I am a straight man.", "You're killing it, amazing performance!", "You are an idiot."]:
    display(HTML(render_html(*explain(clf, s))))

## 5. Optional: open the full dashboard

Starts the Streamlit app in the background and prints a public `trycloudflare.com` link within about a minute. Anyone with the link can use it, so run the next cell when you're done to stop it.

In [ ]:
import re, subprocess, time, urllib.request

subprocess.run("pkill -f '[s]treamlit run'; pkill -f '[c]loudflared tunnel'", shell=True)
subprocess.run("test -f cloudflared || (wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared)", shell=True, check=True)

# 1. Start Streamlit in the background and wait until it answers
subprocess.Popen(["streamlit", "run", "app/app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        urllib.request.urlopen("http://localhost:8501", timeout=2)
        print("Streamlit is running")
        break
    except Exception:
        time.sleep(1)
else:
    print(open("streamlit.log").read()[-3000:])
    raise RuntimeError("Streamlit did not start; its log is printed above")

# 2. Open a public tunnel and print its link
subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8501"],
                 stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    log = open("cloudflared.log").read()
    link = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", log)
    if link or "failed to request" in log:
        break
    time.sleep(1)
if link:
    print("Dashboard:", link.group(0))
else:
    print(log[-3000:])
    raise RuntimeError("No tunnel link; the cloudflared log is printed above")

In [ ]:
# Check that the dashboard is still running (both processes should be listed)
!pgrep -af '[s]treamlit run|[c]loudflared tunnel'
!tail -n 5 cloudflared.log

In [ ]:
# Stop the dashboard and its public link.
# Left commented out so that "Run all" does not stop the dashboard right after starting it.
# Remove the # from the next line and run this cell when you are done.
# !pkill -f '[s]treamlit run'; pkill -f '[c]loudflared tunnel'; echo stopped